# Creating Dataset

In [2]:
# ============================================================
# DATASET + AUGMENTATION PIPELINE (InferenceSequence PKL)
# ============================================================

import os
import pickle
import numpy as np
import json
from typing import Dict, Tuple, List
from scipy.interpolate import interp1d
from landmarkers.inferences import InferenceSequence
from landmarkers.landmarks import LandmarksSequence

# ------------------------------------------------------------
# Cargar configuración desde config.json
# ------------------------------------------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

dataset_config = config['create_dataset']
ACTIONS = dataset_config['actions']
SEQUENCE_LENGTH = dataset_config['sequence_length']
DATA_PATH = dataset_config['data_path']
OUTPUT_PATH = dataset_config['output_path']
augmentation_counts: Dict[str, int] = dataset_config['augmentation_counts']
aug_config = dataset_config['augmentation']
NOISE_STD = aug_config['noise_std']
SCALE_RANGE = tuple(aug_config['scale_range'])
DROPOUT_PROB = aug_config['dropout_prob']
TEMPORAL_PROB = aug_config['temporal_prob']

# ------------------------------------------------------------
# Augmentaciones espaciales
# ------------------------------------------------------------
def add_spatial_noise(seq: np.ndarray, std: float) -> np.ndarray:
	return seq + np.random.normal(0, std, seq.shape)

def scale_sequence(seq: np.ndarray, scale: float) -> np.ndarray:
	return seq * scale

def frame_dropout(seq: np.ndarray, idx: int) -> np.ndarray:
	seq = np.array(seq)
	seq[idx] = 0
	return seq

# ------------------------------------------------------------
# Augmentaciones temporales
# ------------------------------------------------------------
def temporal_interpolation(
	seq: np.ndarray,
	sub_start: int,
	sub_end: int,
	sequence_length: int
) -> np.ndarray:
	sub_seq = seq[sub_start:sub_end]
	x_old = np.linspace(0, 1, num=len(sub_seq))
	x_new = np.linspace(0, 1, num=sequence_length)
	f = interp1d(x_old, sub_seq, axis=0)
	return f(x_new)

def temporal_padding(
	seq: np.ndarray,
	sub_start: int,
	sub_end: int,
	sequence_length: int
) -> np.ndarray:
	sub_seq = seq[sub_start:sub_end]
	pad_len = sequence_length - len(sub_seq)
	pad_front = pad_len // 2
	pad_back = pad_len - pad_front
	return np.vstack([
		np.tile(sub_seq[0], (pad_front, 1, 1)),
		sub_seq,
		np.tile(sub_seq[-1], (pad_back, 1, 1))
	])

# ------------------------------------------------------------
# Pipeline de augmentación principal
# ------------------------------------------------------------
def augment_sequence(
	seq: np.ndarray,
	sequence_length: int,
	noise_std: float = NOISE_STD,
	scale_range: Tuple[float, float] = SCALE_RANGE,
	dropout_prob: float = DROPOUT_PROB,
	temporal_prob: float = TEMPORAL_PROB
) -> np.ndarray:
	seq = np.array(seq)
	orig_len = len(seq)

	# Ruido espacial
	if np.random.rand() < 1.0:
		seq = add_spatial_noise(seq, noise_std)

	# Escalado
	if np.random.rand() < 1.0:
		scale = np.random.uniform(*scale_range)
		seq = scale_sequence(seq, scale)

	# Frame dropout
	if np.random.rand() < dropout_prob:
		idx = np.random.randint(0, orig_len)
		seq = frame_dropout(seq, idx)

	# Transformaciones temporales
	if np.random.rand() < temporal_prob:
		sub_start = np.random.randint(0, orig_len // 2)
		sub_end = sub_start + np.random.randint(orig_len // 2, orig_len)
		sub_end = min(sub_end, orig_len)

		if np.random.rand() < 0.5:
			seq = temporal_interpolation(seq, sub_start, sub_end, sequence_length)
		else:
			seq = temporal_padding(seq, sub_start, sub_end, sequence_length)

	return seq.astype(np.float32)

# ------------------------------------------------------------
# Carga de secuencias desde PKL
# ------------------------------------------------------------
def load_sequence_from_pkl(pkl_path: str) -> np.ndarray:
	"""
	Devuelve array shape (T, 21, 3)
	"""
	with open(pkl_path, "rb") as f:
		infrence_sequence: InferenceSequence = pickle.load(f)  # InferenceSequence

	landmark_sequence: LandmarksSequence = infrence_sequence.landmarks_sequence
	landmark_sequence = landmark_sequence.resample()
	landmark_sequence = landmark_sequence.centered(0)
	frames = landmark_sequence.array
	return np.array(frames, dtype=np.float32)

def combine_hands(seq_right: np.ndarray, seq_left: np.ndarray) -> np.ndarray:
	"""
	(T,21,3) + (T,21,3) -> (T,42,3)
	"""
	return np.concatenate([seq_right, seq_left], axis=1)

# ------------------------------------------------------------
# Construcción del dataset
# ------------------------------------------------------------
label_map: Dict[str, int] = {label: i for i, label in enumerate(ACTIONS)}

sequences: List[np.ndarray] = []
labels: List[int] = []

for action in ACTIONS:
	action_path = os.path.join(DATA_PATH, action)
	if not os.path.isdir(action_path):
		continue

	for seq_folder in os.listdir(action_path):
		seq_path = os.path.join(action_path, seq_folder)
		if not os.path.isdir(seq_path):
			continue

		right_pkl = os.path.join(seq_path, "right.pkl")
		left_pkl = os.path.join(seq_path, "left.pkl")

		if not (os.path.exists(right_pkl) and os.path.exists(left_pkl)):
			continue

		# Cargar secuencias
		seq_right = load_sequence_from_pkl(right_pkl)  # (T,21,3)
		seq_left = load_sequence_from_pkl(left_pkl)    # (T,21,3)

		# Combinar manos
		window = combine_hands(seq_right, seq_left)    # (T,42,3)

		# Secuencia original
		sequences.append(window)
		labels.append(label_map[action])

		# Augmentaciones
		n_aug = augmentation_counts.get(action, 1)
		for _ in range(n_aug):
			aug = augment_sequence(
				window,
				sequence_length=SEQUENCE_LENGTH,
				temporal_prob=0
			)
			sequences.append(aug)
			labels.append(label_map[action])

# ------------------------------------------------------------
# Guardado del dataset final
# ------------------------------------------------------------
X = np.stack(sequences)
y = np.array(labels, dtype=np.int32)

np.savez_compressed(
	OUTPUT_PATH,
	X=X,
	y=y
)

print("✅ Dataset guardado correctamente")
print("X.shape =", X.shape)
print("y.shape =", y.shape)


✅ Dataset guardado correctamente
X.shape = (2775, 10, 42, 3)
y.shape = (2775,)
